# Lecture17: KuCoin NEAR Basis RL Bot\n\nThis notebook demonstrates an end-to-end flow:\n1. Load KuCoin market history (spot + futures).\n2. Build basis features (volume, volatility, z-score, momentum).\n3. Train a Q-learning policy with baseline guidance.\n4. Run one paper-mode live decision tick.

In [ ]:
from pathlib import Path\nimport sys\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\n\nrepo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()\nsrc_path = str((repo_root / 'src').resolve())\nif src_path not in sys.path:\n    sys.path.insert(0, src_path)\n\nfrom kucoin_near_basis_rl.runtime_env import load_env_file\nfrom kucoin_near_basis_rl.train import run_training\nfrom kucoin_near_basis_rl.live import run_live\n\nprint('repo_root =', repo_root)

In [ ]:
env_file = repo_root / '.runtime' / 'kucoin.env'\nloaded = load_env_file(env_file, overwrite=False)\nprint('Loaded env vars from', env_file if env_file.exists() else '(missing)')\nprint('Loaded keys:', sorted(loaded.keys()))\n\nconfig_path = repo_root / 'config' / 'micro_near_v1_1m.json'\nmodel_path = repo_root / 'models' / 'near_basis_qlearning.json'\nfeatures_path = repo_root / 'reports' / 'near_basis_features.csv'\nmodel_path.parent.mkdir(parents=True, exist_ok=True)\nfeatures_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
metrics = run_training(\n    config_path=str(config_path),\n    model_out=str(model_path),\n    features_out=str(features_path),\n)\nmetrics

In [ ]:
df = pd.read_csv(features_path, parse_dates=['timestamp'])\ndisplay(df.tail(5))\n\nfig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)\naxes[0].plot(df['timestamp'], df['basis'], label='basis', color='tab:blue')\naxes[0].axhline(0.0, color='black', linewidth=1)\naxes[0].set_title('Basis')\naxes[0].legend()\n\naxes[1].plot(df['timestamp'], df['basis_zscore'], label='z-score', color='tab:red')\naxes[1].axhline(0.0, color='black', linewidth=1)\naxes[1].axhline(1.8, color='gray', linestyle='--', linewidth=1)\naxes[1].axhline(-1.8, color='gray', linestyle='--', linewidth=1)\naxes[1].set_title('Basis Z-score')\naxes[1].legend()\nplt.tight_layout()\nplt.show()

In [ ]:
run_live(\n    config_path=str(config_path),\n    model_path=str(model_path),\n    paper=True,\n    once=True,\n)

## Live run (real orders)\n\nUse script mode from terminal:\n```\npython run_trade_signal.py --mode live --run-real-order --config config/micro_near_v1_1m.json --model-path models/near_basis_qlearning.json\n```\n\nMake sure `.runtime/kucoin.env` has valid `KUCOIN_API_KEY`, `KUCOIN_API_SECRET`, `KUCOIN_API_PASSPHRASE`.